### ======================================================
###     Pore Network Model Metrics Extraction Tool 
### Using SNOW2 and OpenPNM from Segmented 3D Volumes
### ======================================================

In [1]:
# ██████████████████████████████████████████████████████████████████████
# █  PORE NETWORK MODEL METRICS EXTRACTION TOOL                        █
# █  USING SNOW2 AND OPENPNM FROM SEGMENTED 3D VOLUMES                 █
# █                                                                    █
# █  Author: Leonardo Almeida de Campos                                █
# █  Karlsruhe Institute of Technology (KIT)-ITCP                      █
# █  AI-assisted: OpenAI ChatGPT (GPT-5.3)                             █
# █  04.2026                                                           █
# ██████████████████████████████████████████████████████████████████████

In [ ]:
### Pore Network Modeling (PNM) from 3-label TIFF data (0 = outside, 1 = solid, 2 = pores)

# This notebook performs:
# 1. Loading of a 3D labeled TIFF dataset (Avizo export).
# 2. Validation of label structure and basic preprocessing.
# 3. Cropping to the particle region of interest.
# 4. Construction and optional cleaning of the pore mask.
# 5. Network extraction using PoreSpy (SNOW2 algorithm).
# 6. Import of the pore network into OpenPNM.
# 7. Calculation of structural and transport properties:
#    - Porosity
#    - Throat size distribution
#    - Pore size distribution (PSD)
#    - Coordination number distribution
# 8. Optional 3D visualization using PyVista.
# 9. Export of figures and computed results to disk.

##### =====================================================
#### Libraries
##### =====================================================

In [ ]:
import sys
import os
from pathlib import Path
from datetime import datetime
import numpy as np
import tifffile as tiff
import porespy as ps
import openpnm as op
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import dijkstra, connected_components
import ipywidgets as widgets
from IPython.display import display, clear_output
import plotly.graph_objects as go
from scipy import ndimage as ndi
from plotly.subplots import make_subplots

try:
    import pyvista as pv
    _HAVE_PYVISTA = True
except Exception:
    _HAVE_PYVISTA = False

print("PyVista available:", _HAVE_PYVISTA)
print("libraries available")

##### =====================================================
##### SETTINGS
##### =====================================================

In [ ]:
sample_name = # "Cat_x_labels.tif" # <-- sample name here
BASE_DIR = # Path("D:/Data/raw/cat_x")  # <-- change input directory
OUTPUT_DIR = # Path("D:/Data/processed") / cat_x  # <-- change output directory

VOXEL_SIZE_NM = 99.78                # e.g. 99.78 nm
resolution_filter_nm = 473.14        # e.g. 473.14 <-- set ONCE here (nm)

FLOW_AXIS = 2                        # 0=z, 1=y, 2=x

# For 3-label volumes: 0=outside, 1=solid, 2=pores
LABEL_OUTSIDE = 0
LABEL_SOLID = 1
LABEL_PORES = 2

CLEAN_OPENING = True
OPENING_SIZE = 2
SHOW_TOP_CANDIDATES = 5

# ---- 3D plotting controls ----
DO_3D = True                         # master switch for 3D outputs
DO_3D_OVERLAY = True                 # pores surface + network overlay
DO_3D_DISTMAP = True                 # distance-to-surface map
PV_DOWNSAMPLE = 2                    # 2 or 3 for speed; 1 for full detail
PORE_SURF_OPACITY = 0.10             # pores surface opacity in overlay
TUBE_RADIUS_SCALE = 0.25             # tube radius = scale * median(throat radius)
DIST_ISO_VALUES_UM = (0.5, 1.0, 2.0, 5.0)  # iso-shells for distance map (µm)

# ---- Interactive 3D ----
PV_INTERACTIVE = True                # True = interactive window; False = off-screen PNGs
PV_PICKING = True                    # click to print picked coordinates (overlay view)
PV_TOGGLES = True                    # key toggles: 1=surface, 2=throats, 3=pores
PV_OPACITY_SLIDER = True             # slider to adjust pore surface opacity live
PV_SAVE_SCREENSHOT_ON_SHOW = False   # if True, saves a screenshot even in interactive mode
PV_DECIMATE_SURFACE = 0.0            # 0.0 disables; else 0..1 fraction to reduce (e.g. 0.8 reduces 80%)
PV_SMOOTH_SURFACE_ITERS = 0          # 0 disables; e.g. 20 for mild smoothing

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR.resolve())


##### =====================================================
##### HELPERS: Main functions
##### =====================================================

In [ ]:
# =====================================================
# HELPERS
# =====================================================

def load_tiff_with_nice_errors(path: Path) -> np.ndarray:
    try:
        return tiff.imread(str(path))
    except ValueError as e:
        if "requires the 'imagecodecs' package" in str(e):
            raise RuntimeError(
                "Your TIFF is compressed (e.g., LZW). Install decoder in your venv and rerun:\n\n"
                "  pip install imagecodecs\n"
            ) from e
        raise


def run_snow2(pores: np.ndarray, voxel_size_m: float):
    """Compatibility wrapper for different PoreSpy versions."""
    try:
        return ps.networks.snow2(pores, voxel_size=voxel_size_m)
    except TypeError:
        try:
            return ps.networks.snow2(image=pores, voxel_size=voxel_size_m)
        except TypeError as e:
            raise RuntimeError("Could not call porespy.networks.snow2 with known signatures") from e


def get_prop(pn, keys):
    for k in keys:
        if k in pn.keys():
            return pn[k], k
    raise KeyError(f"None of these properties found on network: {keys}")


def bbox_slices(mask: np.ndarray, pad=2):
    coords = np.argwhere(mask)
    if coords.size == 0:
        raise RuntimeError("Mask is empty; cannot compute bounding box")
    z0, y0, x0 = coords.min(axis=0)
    z1, y1, x1 = coords.max(axis=0) + 1
    z0 = max(z0 - pad, 0); y0 = max(y0 - pad, 0); x0 = max(x0 - pad, 0)
    z1 = min(z1 + pad, mask.shape[0]); y1 = min(y1 + pad, mask.shape[1]); x1 = min(x1 + pad, mask.shape[2])
    return (slice(z0, z1), slice(y0, y1), slice(x0, x1))


def domain_length_and_area_from_particle(particle_mask: np.ndarray, voxel_size_m: float, axis: int):
    """
    L = extent of particle bounding box along axis (m)
    A = average particle cross-sectional area along axis (m^2)
    """
    coords = np.argwhere(particle_mask)
    mins = coords.min(axis=0)
    maxs = coords.max(axis=0) + 1
    L_vox = float(maxs[axis] - mins[axis])
    L = L_vox * voxel_size_m

    if axis == 0:
        areas = particle_mask.sum(axis=(1, 2))
    elif axis == 1:
        areas = particle_mask.sum(axis=(0, 2))
    else:
        areas = particle_mask.sum(axis=(0, 1))

    areas = areas[areas > 0]
    if areas.size == 0:
        raise RuntimeError("Particle mask has no nonzero cross-sections along axis")
    A = float(np.mean(areas)) * (voxel_size_m ** 2)
    return L, A


def axis_labels(axis: int):
    if axis == 2:
        return "xmin", "xmax"
    if axis == 1:
        return "ymin", "ymax"
    return "zmin", "zmax"


def pore_connected_components(pn):
    conns = pn["throat.conns"]
    if conns.size == 0:
        return np.arange(pn.Np, dtype=int)

    i = conns[:, 0].astype(int)
    j = conns[:, 1].astype(int)

    data = np.ones(i.size * 2, dtype=np.uint8)
    row = np.hstack([i, j])
    col = np.hstack([j, i])

    G = coo_matrix((data, (row, col)), shape=(pn.Np, pn.Np)).tocsr()
    _, labels_cc = connected_components(csgraph=G, directed=False, return_labels=True)
    return labels_cc.astype(int)


def percolating_subnetwork(pn, inlet_pores, outlet_pores):
    cnum = pore_connected_components(pn)

    inlet_clusters = set(cnum[inlet_pores].tolist())
    outlet_clusters = set(cnum[outlet_pores].tolist())
    keep_clusters = inlet_clusters.intersection(outlet_clusters)

    if len(keep_clusters) == 0:
        raise RuntimeError(
            "No percolating cluster found between inlet and outlet. "
            "Try changing FLOW_AXIS or check if pores connect across the ROI."
        )

    keep_mask = np.isin(cnum, list(keep_clusters))
    pores_to_trim = np.where(~keep_mask)[0]

    pn2 = op.network.Network()
    pn2.update(pn)
    op.topotools.trim(network=pn2, pores=pores_to_trim)

    print(f"Clusters total: {len(np.unique(cnum))}")
    print(f"Percolating clusters kept: {len(keep_clusters)}")
    print(f"Pores kept: {pn2.Np} / {pn.Np}")
    print(f"Throats kept: {pn2.Nt} / {pn.Nt}")

    return pn2


def adjacency_matrix_from_network(pn, throat_length_key="throat.total_length"):
    conns = pn["throat.conns"]
    if throat_length_key not in pn.keys():
        raise KeyError(f"{throat_length_key} not found on network")

    w = pn[throat_length_key].astype(float)
    i = conns[:, 0]
    j = conns[:, 1]

    data = np.hstack([w, w])
    row = np.hstack([i, j])
    col = np.hstack([j, i])

    A = coo_matrix((data, (row, col)), shape=(pn.Np, pn.Np)).tocsr()
    return A


def geometric_tortuosity_shortest_path(pn_flow, axis=2, throat_length_key="throat.total_length"):
    in_lab, out_lab = axis_labels(axis)
    inlet = pn_flow.pores(in_lab)
    outlet = pn_flow.pores(out_lab)

    if inlet.size == 0 or outlet.size == 0:
        raise RuntimeError("No inlet/outlet pores found for geometric tortuosity")

    coords = pn_flow["pore.coords"]
    straight_L = float(np.ptp(coords[:, axis]))

    A = adjacency_matrix_from_network(pn_flow, throat_length_key=throat_length_key)
    dist = dijkstra(csgraph=A, directed=False, indices=inlet)
    dmin = np.min(dist[:, outlet], axis=1)
    dmin = dmin[np.isfinite(dmin)]
    if dmin.size == 0:
        raise RuntimeError("No finite paths from inlet to outlet found")

    tau_geo = dmin / straight_L
    return tau_geo, straight_L


def dead_end_stats(pn_flow, exclude_boundary=True):
    z = pn_flow.num_neighbors(pn_flow.Ps)
    dead = (z == 1)
    if exclude_boundary and "pore.boundary" in pn_flow.keys():
        dead = dead & ~pn_flow["pore.boundary"]

    dead_ids = pn_flow.Ps[dead]
    frac_count = dead_ids.size / pn_flow.Np

    v = pn_flow["pore.volume"].astype(float)
    frac_vol = float(np.sum(v[dead_ids]) / np.sum(v))

    return dead_ids, float(frac_count), float(frac_vol)


def add_diffusive_conductance_cylinders(
    pn_flow,
    phase,
    D=1.0,
    throat_area_key="throat.cross_sectional_area",
    throat_len_key="throat.total_length",
):
    if throat_area_key not in pn_flow.keys():
        raise KeyError(f"Missing {throat_area_key} on network")
    if throat_len_key not in pn_flow.keys():
        raise KeyError(f"Missing {throat_len_key} on network")

    A = pn_flow[throat_area_key].astype(float)
    L = pn_flow[throat_len_key].astype(float)
    L = np.clip(L, 1e-30, np.inf)

    if np.isscalar(D):
        D_arr = float(D) * np.ones(pn_flow.Nt, dtype=float)
    else:
        D_arr = np.asarray(D, dtype=float)
        if D_arr.size != pn_flow.Nt:
            raise ValueError("If D is array, it must have length Nt")

    g = D_arr * A / L
    g[~np.isfinite(g)] = 0.0

    phase["throat.diffusive_conductance"] = g
    return g


def add_hydraulic_conductance_cylinders(
    pn_flow,
    phase,
    mu=1e-3,
    throat_diam_key="throat.equivalent_diameter",
    throat_len_key="throat.total_length",
):
    if throat_diam_key not in pn_flow.keys():
        raise KeyError(f"Missing {throat_diam_key} on network")
    if throat_len_key not in pn_flow.keys():
        raise KeyError(f"Missing {throat_len_key} on network")

    d = pn_flow[throat_diam_key].astype(float)
    L = pn_flow[throat_len_key].astype(float)

    L = np.clip(L, 1e-30, np.inf)
    r = np.clip(d / 2.0, 0.0, np.inf)

    g = (np.pi * r**4) / (8.0 * mu * L)
    g[~np.isfinite(g)] = 0.0

    phase["throat.hydraulic_conductance"] = g
    return g


def calc_perm_from_stokesflow(sf, pn_flow, phase, inlet_pores, L, A, dP=1.0):
    if "throat.viscosity" in phase.keys():
        mu = float(np.mean(phase["throat.viscosity"]))
    elif "pore.viscosity" in phase.keys():
        mu = float(np.mean(phase["pore.viscosity"]))
    else:
        raise KeyError("No viscosity found on phase")

    try:
        q_in = sf.rate(pores=inlet_pores)
        q_in = np.asarray(q_in, dtype=float)
        Q = float(np.sum(np.abs(q_in)))
        if not np.isfinite(Q) or Q == 0.0:
            raise RuntimeError
    except Exception:
        conns = pn_flow["throat.conns"]
        inlet_set = set(inlet_pores.tolist())
        tmask = np.array([(int(i) in inlet_set) ^ (int(j) in inlet_set) for i, j in conns], dtype=bool)
        tidx = np.where(tmask)[0]
        if tidx.size == 0:
            raise RuntimeError("No inlet-connecting throats found to compute Q")
        Q = 0.0
        for t in tidx:
            qt = sf.rate(throats=[int(t)])
            Q += float(np.abs(qt))

    K = (Q * mu * L) / (A * dP)
    return K, Q, mu


def voxel_percolating_porosity(pores: np.ndarray, particle: np.ndarray, axis: int) -> float:
    pores_in = pores & particle
    lab, n = ndi.label(pores_in)
    if n == 0:
        return 0.0

    if axis == 0:
        pore_min = lab[0, :, :]
        pore_max = lab[-1, :, :]
    elif axis == 1:
        pore_min = lab[:, 0, :]
        pore_max = lab[:, -1, :]
    else:
        pore_min = lab[:, :, 0]
        pore_max = lab[:, :, -1]

    s_min = set(np.unique(pore_min[pore_min > 0]).tolist())
    s_max = set(np.unique(pore_max[pore_max > 0]).tolist())
    keep = s_min.intersection(s_max)
    if len(keep) == 0:
        return 0.0

    keep_mask = np.isin(lab, list(keep))
    phi_perc = float(keep_mask.sum() / particle.sum())
    return phi_perc


def transport_tortuosity_diffusion(pn_flow, domain_area, domain_length, axis=2, D_bulk=1.0, phi_for_tau=None):
    in_lab, out_lab = axis_labels(axis)
    inlet = pn_flow.pores(in_lab)
    outlet = pn_flow.pores(out_lab)
    if inlet.size == 0 or outlet.size == 0:
        raise RuntimeError("No inlet/outlet pores found for diffusion tortuosity")

    pn_flow.add_model_collection(op.models.collections.geometry.spheres_and_cylinders)
    pn_flow.regenerate_models()

    phase = op.phase.Phase(network=pn_flow)
    phase["pore.diffusivity"] = D_bulk
    phase["throat.diffusivity"] = D_bulk

    add_diffusive_conductance_cylinders(
        pn_flow=pn_flow,
        phase=phase,
        D=D_bulk,
        throat_area_key="throat.cross_sectional_area",
        throat_len_key="throat.total_length",
    )

    fd = op.algorithms.FickianDiffusion(network=pn_flow, phase=phase)
    fd.set_value_BC(pores=inlet, values=1.0)
    fd.set_value_BC(pores=outlet, values=0.0)
    fd.run()

    if hasattr(fd, "calc_effective_diffusivity"):
        D_eff = float(fd.calc_effective_diffusivity(domain_area=domain_area, domain_length=domain_length))
    else:
        q_in = np.asarray(fd.rate(pores=inlet), dtype=float)
        J = float(np.sum(np.abs(q_in)))
        D_eff = (J * domain_length) / (domain_area * 1.0)

    if phi_for_tau is None:
        domain_vol = domain_area * domain_length
        phi_used = float(np.sum(pn_flow["pore.volume"]) / domain_vol)
    else:
        phi_used = float(phi_for_tau)

    tau = float((phi_used * D_bulk) / D_eff)
    return float(D_eff), float(phi_used), float(tau)


##### =====================================================
##### HELPERS FOR 3D PLOTTING 
##### =====================================================

In [ ]:
def pv_setup_jupyter(backend="trame"):
    """
    Configure PyVista rendering for Jupyter.

    backend:
      - "trame": interactive inline (recommended)
      - "static": static images inline
    """
    if not _HAVE_PYVISTA:
        raise RuntimeError("PyVista not available. Install: pip install pyvista vtk")

    import pyvista as pv
    pv.set_plot_theme("document")
    pv.set_jupyter_backend(backend)
    pv.global_theme.jupyter_backend = backend
    print(f"PyVista Jupyter backend set to: {backend}")


def add_screenshot_hotkey(plotter, default_dir: Path, key="k"):
    """
    Press 'k' in the PyVista window to choose a screenshot path (Save As...).
    In *true notebook inline* (trame) this may not be super useful, but it still works
    when an external rendering window is available.
    """
    def _do_screenshot():
        default_dir.mkdir(parents=True, exist_ok=True)
        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        fallback = default_dir / f"pyvista_screenshot_{ts}.png"

        try:
            import tkinter as tk
            from tkinter import filedialog

            root = tk.Tk()
            root.withdraw()
            try:
                root.attributes("-topmost", True)
            except Exception:
                pass

            path = filedialog.asksaveasfilename(
                initialdir=str(default_dir.resolve()),
                initialfile=fallback.name,
                defaultextension=".png",
                filetypes=[("PNG image", "*.png")],
                title="Save PyVista screenshot",
            )
            root.destroy()

            if not path:
                print("Screenshot canceled.")
                return

            plotter.screenshot(path)
            print("Saved screenshot:", path)

        except Exception as e:
            plotter.screenshot(str(fallback))
            print(f"[tkinter unavailable -> fallback] Saved screenshot: {fallback} ({e})")

    plotter.add_key_event(key.lower(), _do_screenshot)
    plotter.add_key_event(key.upper(), _do_screenshot)


def pv_grid_from_mask(mask_zyx: np.ndarray, spacing_m: float):
    """
    Build PyVista ImageData with cell_data['mask'] from a boolean mask (z,y,x).
    """
    import pyvista as pv
    grid = pv.ImageData()
    grid.dimensions = np.array(mask_zyx.shape) + 1
    grid.spacing = (spacing_m, spacing_m, spacing_m)
    grid.origin = (0.0, 0.0, 0.0)
    grid.cell_data["mask"] = mask_zyx.astype(np.uint8).ravel(order="F")
    return grid


def _maybe_simplify_surface(surf):
    if PV_DECIMATE_SURFACE and PV_DECIMATE_SURFACE > 0:
        try:
            surf = surf.decimate_pro(float(PV_DECIMATE_SURFACE))
        except Exception:
            pass
    if PV_SMOOTH_SURFACE_ITERS and PV_SMOOTH_SURFACE_ITERS > 0:
        try:
            surf = surf.smooth(n_iter=int(PV_SMOOTH_SURFACE_ITERS))
        except Exception:
            pass
    return surf


def plot_network_overlay_3d(
    pores_mask_zyx: np.ndarray,
    pn_net,
    voxel_size_m: float,
    ds: int,
    out_png: Path,
    pores_opacity: float,
    tube_radius_scale: float,
):
    """
    3D overlay:
      - pore isosurface (semi-transparent)
      - throats as tubes
      - pores as spheres

    Fixes:
      - glyph orient warning: orient=False
      - picking deprecation: use_picker=True
    """
    import pyvista as pv

    pores_ds = pores_mask_zyx[::ds, ::ds, ::ds]
    spacing = voxel_size_m * ds

    grid = pv_grid_from_mask(pores_ds, spacing_m=spacing)
    pore_surf = grid.threshold(0.5, scalars="mask").extract_surface().clean()
    pore_surf = _maybe_simplify_surface(pore_surf)

    coords = np.asarray(pn_net["pore.coords"], dtype=float)
    conns = np.asarray(pn_net["throat.conns"], dtype=int)

    # throats as polylines
    lines = np.empty((conns.shape[0], 3), dtype=np.int64)
    lines[:, 0] = 2
    lines[:, 1:] = conns
    lines = lines.ravel()

    poly = pv.PolyData(coords)
    poly.lines = lines

    # tube radius from median throat diameter (fallback: voxel_size)
    tube_radius = None
    for key in ("throat.equivalent_diameter", "throat.inscribed_diameter", "throat.diameter"):
        if key in pn_net.keys():
            td = np.asarray(pn_net[key], dtype=float)
            tube_radius = float(tube_radius_scale) * float(np.nanmedian(td)) / 2.0
            break
    if tube_radius is None or not np.isfinite(tube_radius) or tube_radius <= 0:
        tube_radius = float(tube_radius_scale) * voxel_size_m

    tube = poly.tube(radius=tube_radius, n_sides=12)

    # pore spheres (orient=False avoids "No vector-like data to use for orient")
    pore_pts = pv.PolyData(coords)
    pdiam = None
    for key in ("pore.equivalent_diameter", "pore.inscribed_diameter", "pore.diameter"):
        if key in pn_net.keys():
            pdiam = np.asarray(pn_net[key], dtype=float)
            break
    if pdiam is None:
        pdiam = np.full(pn_net.Np, 2.0 * voxel_size_m, dtype=float)

    pore_pts["pore_radius"] = 0.20 * (pdiam / 2.0)
    sphere_geom = pv.Sphere(theta_resolution=16, phi_resolution=16)
    spheres = pore_pts.glyph(scale="pore_radius", geom=sphere_geom, orient=False)

    p = pv.Plotter(off_screen=not PV_INTERACTIVE)

    if PV_INTERACTIVE:
        add_screenshot_hotkey(p, OUTPUT_DIR, key="k")
        print("\n[3D overlay] Tip: press 'K' to choose where to save a screenshot.")

    a_surf = p.add_mesh(pore_surf, opacity=float(pores_opacity), name="pores_surface")
    p.add_mesh(tube, opacity=0.85, name="throats")
    p.add_mesh(spheres, opacity=0.90, name="pores")

    p.add_axes()
    p.show_grid()
    p.camera_position = "iso"

    # Picking (updated API)
    if PV_INTERACTIVE and PV_PICKING:
        def _picked_point(point):
            print("Picked point (x,y,z):", point)
        p.enable_point_picking(callback=_picked_point, show_message=True, use_picker=True)

    # Key toggles
    if PV_INTERACTIVE and PV_TOGGLES:
        def _toggle_actor(name):
            try:
                actor = p.renderer._actors.get(name)
                if actor is None:
                    return
                actor.SetVisibility(not actor.GetVisibility())
                p.render()
            except Exception:
                pass

        p.add_key_event("1", lambda: _toggle_actor("pores_surface"))
        p.add_key_event("2", lambda: _toggle_actor("throats"))
        p.add_key_event("3", lambda: _toggle_actor("pores"))

    # Opacity slider
    if PV_INTERACTIVE and PV_OPACITY_SLIDER:
        def _set_opacity(val):
            try:
                a_surf.GetProperty().SetOpacity(float(val))
                p.render()
            except Exception:
                pass

        p.add_slider_widget(
            _set_opacity,
            rng=[0.0, 1.0],
            value=float(pores_opacity),
            title="Pore surface opacity",
        )

    if PV_INTERACTIVE:
        if PV_SAVE_SCREENSHOT_ON_SHOW:
            p.show(screenshot=str(out_png))
            print("Saved 3D overlay screenshot:", out_png)
        else:
            p.show()
    else:
        p.show(screenshot=str(out_png))
        print("Saved 3D overlay:", out_png)


def pore_distance_to_particle_surface(labels_zyx: np.ndarray, voxel_size_m: float,
                                      outside_label=0, solid_label=1, pore_label=2):
    outside = labels_zyx == outside_label
    particle = (labels_zyx == solid_label) | (labels_zyx == pore_label)
    pores = labels_zyx == pore_label

    outside_dil = ndi.binary_dilation(outside, structure=np.ones((3, 3, 3), bool))
    surface = particle & outside_dil
    if not np.any(surface):
        raise RuntimeError("No particle surface detected (check labels or cropping).")

    dist_vox = ndi.distance_transform_edt(~surface)
    dist_m = dist_vox * voxel_size_m
    return pores & particle, dist_m, surface


def plot_pore_distance_map_3d(labels_zyx: np.ndarray, voxel_size_m: float, ds: int,
                              out_png: Path, iso_values_um=()):
    import pyvista as pv
    pv.set_plot_theme("document")

    pores_mask, dist_m, _surface = pore_distance_to_particle_surface(
        labels_zyx, voxel_size_m,
        outside_label=LABEL_OUTSIDE, solid_label=LABEL_SOLID, pore_label=LABEL_PORES
    )

    pores_ds = pores_mask[::ds, ::ds, ::ds]
    dist_ds_um = (dist_m[::ds, ::ds, ::ds] * 1e6).astype(np.float32)
    spacing = voxel_size_m * ds

    grid = pv.ImageData()
    grid.dimensions = np.array(pores_ds.shape) + 1
    grid.spacing = (spacing, spacing, spacing)
    grid.origin = (0.0, 0.0, 0.0)

    grid.cell_data["pores"] = pores_ds.astype(np.uint8).ravel(order="F")
    grid.cell_data["dist_um"] = dist_ds_um.ravel(order="F")

    pore_surf = grid.threshold(0.5, scalars="pores").extract_surface().clean()
    pore_surf = _maybe_simplify_surface(pore_surf)

    p = pv.Plotter(off_screen=not PV_INTERACTIVE)

    if PV_INTERACTIVE:
        add_screenshot_hotkey(p, OUTPUT_DIR, key="k")
        print("\n[3D distmap] Tip: press 'K' to choose where to save a screenshot.")

    p.add_mesh(pore_surf, scalars="dist_um", opacity=0.85, show_scalar_bar=True, name="dist_surface")

    if iso_values_um:
        masked = grid.copy()
        dist_um = masked.cell_data["dist_um"].copy()
        dist_um[masked.cell_data["pores"] == 0] = 0.0
        masked.cell_data["dist_um_masked"] = dist_um
        for v in iso_values_um:
            try:
                shell = masked.contour(isosurfaces=[float(v)], scalars="dist_um_masked").extract_surface().clean()
                shell = _maybe_simplify_surface(shell)
                if shell.n_points > 0:
                    p.add_mesh(shell, opacity=0.25)
            except Exception:
                pass

    p.add_axes()
    p.show_grid()
    p.camera_position = "iso"

    if PV_INTERACTIVE:
        if PV_SAVE_SCREENSHOT_ON_SHOW:
            p.show(screenshot=str(out_png))
            print("Saved 3D distance map screenshot:", out_png)
        else:
            p.show()
    else:
        p.show(screenshot=str(out_png))
        print("Saved 3D distance map:", out_png)

#============================================================================

# Set Trame environment
pv_setup_jupyter("trame")

#### Step 1 - Find + load TIF.

In [ ]:
print(f"Searching for TIFF in: {BASE_DIR}")
tiff_path = BASE_DIR / sample_name
print("Using TIFF:", tiff_path)

print("Loading 3D TIFF...")
labels = load_tiff_with_nice_errors(tiff_path).astype(np.int32)

if labels.ndim != 3:
    raise RuntimeError(f"Expected a 3D volume, got shape {labels.shape}")

uniq, counts = np.unique(labels, return_counts=True)
print("Unique labels (value: count), first 30:")
for v, c in list(zip(uniq, counts))[:30]:
    print(f"  {int(v)}: {int(c)}")

print("Loaded volume shape (z,y,x):", labels.shape)


#### Step 2 - Build masks + crop to particle ROI.

In [ ]:
# ------------------------------------------------------------------------------------
# Resolution for voxel size -- Convert voxel size to meters for transport calculations
# ------------------------------------------------------------------------------------

voxel_size_m = VOXEL_SIZE_NM * 1e-9

# -----------------------------------------------------
# 3-label mode: 0 outside, 1 solid, 2 pores
# -----------------------------------------------------
u = set(np.unique(labels).tolist())
if not u.issuperset({LABEL_OUTSIDE, LABEL_SOLID, LABEL_PORES}):
    raise RuntimeError(
        "Expected 3-label volume with labels {0,1,2} meaning {outside, solid, pores}."
    )

print("Detected 3-label volume: 0=outside, 1=solid, 2=pores")

particle = (labels == LABEL_SOLID) | (labels == LABEL_PORES)
pores = (labels == LABEL_PORES)

# Crop to particle bounding box
slc = bbox_slices(particle, pad=2)
labels = labels[slc]
particle = particle[slc]
pores = pores[slc]

# -----------------------------------------------------
# Optional cleaning ONLY on pore mask (inside particle)
# -----------------------------------------------------

pores = pores & particle
if CLEAN_OPENING:
    pores = ndi.binary_opening(pores, structure=np.ones((OPENING_SIZE, OPENING_SIZE, OPENING_SIZE)))
    pores = pores & particle

phi_vox = float(pores.sum() / particle.sum())
phi_perc_vox = voxel_percolating_porosity(pores, particle, axis=FLOW_AXIS)

print("Cropped shape (z,y,x):", labels.shape)
print(f"Voxel porosity inside particle (total): {phi_vox:.4f}")
print(f"Voxel percolating porosity (axis={FLOW_AXIS}): {phi_perc_vox:.4f}")


#### Step 3 — Quick sanity-check slices (visualize the middle slice).

In [ ]:
mid = labels.shape[0] // 2

plt.figure(figsize=(6, 5))
plt.imshow(labels[mid], cmap="tab20")
#plt.colorbar(label="Label ID")
plt.title("Middle slice (labels, cropped to particle)")
plt.axis("off")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "labels_middle_slice.png", dpi=300)
plt.show()

plt.figure(figsize=(6, 5))
plt.imshow(pores[mid], cmap="gray")
plt.title("Middle slice (pore mask, cropped)")
plt.axis("off")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "pore_mask_middle_slice.png", dpi=300)
plt.show()

print("Saved sanity images into:", OUTPUT_DIR.resolve())


#### Step 4 - Resolution Filter.

In [ ]:
def resolution_filter_pores_by_cluster_Deq(pores: np.ndarray, voxel_size_m: float, resolution_filter_nm: float):
    """
    Removes pore clusters whose spherical-equivalent diameter is < resolution_filter_nm.
    Returns: pores_resolved (bool mask), plus some diagnostics.
    """
    pores = np.asarray(pores, dtype=bool)
    pore_labels, N = ndi.label(pores)

    if N == 0:
        raise ValueError("No pore clusters found in 'pores' mask.")

    voxel_volume = float(voxel_size_m) ** 3
    cluster_sizes = ndi.sum(
        np.ones_like(pores, dtype=np.uint8),
        labels=pore_labels,
        index=np.arange(1, N + 1)
    ).astype(float)

    cluster_volumes_m3 = cluster_sizes * voxel_volume
    cluster_Deq_m = (6.0 * cluster_volumes_m3 / np.pi) ** (1.0 / 3.0)
    cluster_Deq_nm = cluster_Deq_m * 1e9

    keep_ids = np.where(cluster_Deq_nm >= float(resolution_filter_nm))[0] + 1
    pores_resolved = np.isin(pore_labels, keep_ids)

    info = {
        "N_clusters_raw": int(N),
        "N_clusters_kept": int(keep_ids.size),
        "voxels_raw": int(pores.sum()),
        "voxels_kept": int(pores_resolved.sum()),
        "keep_ids": keep_ids,
        "cluster_Deq_nm": cluster_Deq_nm,
    }
    return pores_resolved, info


## Applying filter

resolution_filter_nm = resolution_filter_nm # <-- set globaly (check header)
pores_raw = pores.copy()

pores, rf_info = resolution_filter_pores_by_cluster_Deq(
    pores=pores_raw,
    voxel_size_m=voxel_size_m,
    resolution_filter_nm=resolution_filter_nm
)

print(
    f"Resolution filter (Deq >= {resolution_filter_nm} nm): "
    f"clusters {rf_info['N_clusters_raw']} -> {rf_info['N_clusters_kept']}, "
    f"voxels {rf_info['voxels_raw']} -> {rf_info['voxels_kept']}"
)

#### Step 5 - SNOW2 extraction + import to OpenPNM.

In [ ]:
# ----------------------------------------------------------------------
# SNOW2 selected due to robust segmentation of complex pore morphologies
# ----------------------------------------------------------------------

In [ ]:
print("Extracting pore network (SNOW2)...")
snow = run_snow2(pores, voxel_size_m)

print("Importing into OpenPNM...")
pn = op.io.network_from_porespy(snow.network)
print(pn)

# Diameter properties (robust across porespy/openpnm versions)
pore_diam, pore_key = get_prop(pn, ["pore.diameter", "pore.equivalent_diameter", "pore.inscribed_diameter"])
throat_diam, throat_key = get_prop(pn, ["throat.diameter", "throat.equivalent_diameter", "throat.inscribed_diameter"])
print("Using pore size key:", pore_key)
print("Using throat size key:", throat_key)


In [ ]:

# -----------------------
# Precompute once
# -----------------------
coords_m = np.asarray(snow.network["pore.coords"], dtype=float)      # (Np, 3)
conns = np.asarray(snow.network["throat.conns"], dtype=int)          # (Nt, 2)

coords_v = coords_m / voxel_size_m  # voxel indices (z,y,x)
zv, yv, xv = coords_v[:, 0], coords_v[:, 1], coords_v[:, 2]

i_conn = conns[:, 0].astype(int)
j_conn = conns[:, 1].astype(int)

def _max_k_for_axis(axis):
    return snow.regions.shape[axis] - 1


# -----------------------
# Widgets
# -----------------------
axis_widget = widgets.Dropdown(
    options=[("z (axis=0)", 0), ("y (axis=1)", 1), ("x (axis=2)", 2)],
    value=0,
    description="Axis",
)

k_widget = widgets.IntSlider(
    value=min(10, _max_k_for_axis(0)),
    min=0,
    max=_max_k_for_axis(0),
    step=1,
    description="Slice",
    continuous_update=False,
)

tol_widget = widgets.FloatSlider(
    value=1.0,
    min=0.0,
    max=5.0,
    step=0.5,
    description="tol (vox)",
    continuous_update=False,
)

ms_widget = widgets.FloatSlider(
    value=50,
    min=5,
    max=200,
    step=5,
    description="marker",
    continuous_update=False,
)

lw_widget = widgets.FloatSlider(
    value=1.0,
    min=0.2,
    max=3.0,
    step=0.1,
    description="line",
    continuous_update=False,
)

cmap_widget = widgets.Dropdown(
    options=["viridis", "tab20", "nipy_spectral", "turbo"],
    value="viridis",
    description="cmap",
)

out = widgets.Output()


def show_slice(axis=0, k=10, tol=1.0, ms=50, lw=1.0, cmap="viridis"):
    # Background slice + 2D projection mapping
    if axis == 0:
        bg = snow.regions[k, :, :]
        keep = np.abs(zv - k) <= tol
        px, py = xv, yv      # x vs y
    elif axis == 1:
        bg = snow.regions[:, k, :]
        keep = np.abs(yv - k) <= tol
        px, py = xv, zv      # x vs z
    else:
        bg = snow.regions[:, :, k]
        keep = np.abs(xv - k) <= tol
        px, py = yv, zv      # y vs z

    keep_ids = np.where(keep)[0]
    keep_mask = keep

    # Only keep edges fully inside slice band
    ekeep = keep_mask[i_conn] & keep_mask[j_conn]
    ii = i_conn[ekeep]
    jj = j_conn[ekeep]

    with out:
        clear_output(wait=True)

        fig, ax = plt.subplots(figsize=(7, 6))
        ax.imshow(bg.T, cmap=cmap)

        # Draw all line segments in one go (faster than a Python loop)
        # Because bg is transposed, use (py, px) swapped:
        xs = np.vstack([py[ii], py[jj]]).T
        ys = np.vstack([px[ii], px[jj]]).T
        ax.plot(xs.T, ys.T, "k-", lw=lw, alpha=0.9)

        ax.scatter(py[keep], px[keep], s=ms, c="red", edgecolors="none", alpha=0.95)

        ax.axis(False)
        ax.set_title(f"SNOW regions + network overlay | axis={axis} | slice={k} | kept pores={keep_ids.size}")
        plt.tight_layout()
        plt.show()


def refresh(*_):
    show_slice(
        axis=axis_widget.value,
        k=k_widget.value,
        tol=tol_widget.value,
        ms=ms_widget.value,
        lw=lw_widget.value,
        cmap=cmap_widget.value,
    )


def on_axis_change(change):
    axv = change["new"]
    k_widget.max = _max_k_for_axis(axv)
    if k_widget.value > k_widget.max:
        k_widget.value = k_widget.max
    refresh()


# Wire callbacks
axis_widget.observe(on_axis_change, names="value")
for w in (k_widget, tol_widget, ms_widget, lw_widget, cmap_widget):
    w.observe(refresh, names="value")

# Display UI + output
display(widgets.HBox([axis_widget, k_widget]))
display(widgets.HBox([tol_widget, ms_widget, lw_widget, cmap_widget]))
display(out)

# Initial draw
refresh()


#### Step 6 - Basic network statistics plots.

In [ ]:
# -----------------------------
# Font parameters
# -----------------------------
plt.rcParams.update({
    "font.family": "Arial",
    "font.size": 14,
    "axes.labelsize": 14,
    "axes.titlesize": 14,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "legend.fontsize": 14,
    "pdf.fonttype": 42,   # embed TrueType fonts
    "ps.fonttype": 42,
})

ORCHID = "orchid"

pore_diam_nm = pore_diam * 1e9
throat_diam_nm = throat_diam * 1e9
coordination = pn.num_neighbors(pn.Ps)

# -----------------------------
# Pore size distribution
# -----------------------------
plt.figure(figsize=(6, 4))
plt.hist(
    pore_diam_nm,
    bins=60,
    density=True,
    color="#1F45FC",
    edgecolor="#1F45FC",
    linewidth=0.6,
)
plt.xlabel("Pore diameter (nm)")
plt.ylabel("Probability density")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "pore_size_distribution.png", dpi=300, bbox_inches="tight")
plt.show()

# -----------------------------
# Throat size distribution
# -----------------------------
plt.figure(figsize=(6, 4))
plt.hist(
    throat_diam_nm,
    bins=60,
    density=True,
    color="#1F45FC",
    edgecolor="#1F45FC",
    linewidth=0.6,
)
plt.xlabel("Throat diameter (nm)")
plt.ylabel("Probability density")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "throat_size_distribution.png", dpi=300, bbox_inches="tight")
plt.show()

# -----------------------------
# Coordination number
# -----------------------------
plt.figure(figsize=(6, 4))
plt.hist(
    coordination,
    bins=np.arange(coordination.max() + 2) - 0.5,
    color="#1F45FC",
    edgecolor="#1F45FC",
    linewidth=0.6,
)
plt.xlabel("Coordination number")
plt.ylabel("Count")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "coordination_number.png", dpi=300, bbox_inches="tight")
plt.show()

# -----------------------------
# Save raw arrays
# -----------------------------
np.savetxt(OUTPUT_DIR / "pore_diameters_nm.txt", pore_diam_nm)
np.savetxt(OUTPUT_DIR / "throat_diameters_nm.txt", throat_diam_nm)
np.savetxt(OUTPUT_DIR / "S.txt", coordination)

print("Saved PSD + coordination outputs into:", OUTPUT_DIR.resolve())


#### Step 7 - BET comparison (optional).

In [ ]:
# -------------------------
# INPUT (needed for cm3/g)
# -------------------------
m_sample_g = 0.1433  # <-- PUT YOUR SAMPLE MASS IN GRAMS HERE (e.g., 0.050 g)

# -------------------------
# Pore size + pore volume
# -------------------------
D_nm = np.asarray(pore_diam, dtype=float) * 1e9   # pore diameter [nm]

if "pore.volume" in pn.keys():
    V_m3 = np.asarray(pn["pore.volume"], dtype=float)   # pore volume [m^3]
else:
    # fallback: spherical pore assumption
    D_m = D_nm * 1e-9
    V_m3 = (np.pi / 6.0) * D_m**3

mask = np.isfinite(D_nm) & np.isfinite(V_m3) & (D_nm > 0) & (V_m3 > 0)
D_nm = D_nm[mask]
V_m3 = V_m3[mask]

# Convert to cm^3/g (BJH units basis)
V_cm3_g = (V_m3 * 1e6) / float(m_sample_g)   # 1 m^3 = 1e6 cm^3

# -------------------------
# Linear bins in DIAMETER (BJH-style)
# -------------------------
nbins = 60
DV_cm3_g, edges = np.histogram(D_nm, bins=nbins, weights=V_cm3_g)  # DV per bin [cm^3 g^-1]

DD = np.diff(edges)                         # ΔD per bin [nm]
Dm = 0.5 * (edges[:-1] + edges[1:])         # mean diameter per bin [nm]

# -------------------------
# dV/dD in cm3 g^-1 nm^-1
# -------------------------
dV_dD = DV_cm3_g / DD                       # [cm^3 g^-1 nm^-1]

# -------------------------
# Plot dV/dD (BJH units)
# -------------------------
plt.figure(figsize=(6, 4))
plt.plot(Dm, dV_dD, lw=2, color="orchid")
plt.xlabel("Pore diameter, Dm (nm)")
plt.ylabel(r"$dV/dD$ (cm$^3$ g$^{-1}$ nm$^{-1}$)")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "PNM_dVdD_cm3g_nm.png", dpi=300)
plt.show()

# -------------------------
# Save table (BJH)
# -------------------------
np.savetxt(
    OUTPUT_DIR / "PNM_BJH_like_dVdD_table.txt",
    np.column_stack([Dm, DV_cm3_g, DD, dV_dD]),
    header="Dm_nm   DV_cm3_per_g   DD_nm   dVdD_cm3_per_g_per_nm"
)

print("Saved PNM dV/dD (cm^3 g^-1 nm^-1) to:", OUTPUT_DIR.resolve())


#### Step 8 - Macropore identification and spatial distribution mapping.

In [ ]:
# This code defines and executes a routine to quantify and visualize the 3D spatial distribution of macropores inside a segmented 
# catalyst particle. Starting from binary voxel masks for pores and the particle, connected pore clusters are identified in 3D 
# (connected-component labeling) and their sizes are converted to physical volumes using the voxel volume. 
# Each cluster is then characterized by a spherical-equivalent diameter ($D_{\mathrm{eq}}$), 
# and only clusters above a user-defined resolution threshold (in nm) are retained as macropores. For the retained macropores, 
# centroid coordinates are computed and converted to metric units; the particle center of mass and an effective particle radius 
# (maximum distance from the center to any particle voxel) are used to calculate the normalized radial position $r/R$ for each macropore centroid. 
# Marker sizes are scaled with $D_{\mathrm{eq}}$ (linear or square-root scaling) to encode macropore size, while the marker color encodes $r/R$ to 
# represent the macropore proximity to the particle periphery. The routine exports (i) an interactive 3D Plotly visualization 
# (and an optional static PNG) showing macropore centroids within a spherical envelope of the particle, and (ii) a publication-ready histogram
# of the $r/R$ distribution (probability density) using Matplotlib, returning the computed descriptors (IDs, centroids, $D_{\mathrm{eq}}$, $r/R$, 
# and file paths) for downstream analysis.

In [ ]:
def make_main_panel_macropores_plotly(
    pores,                      # voxel pore mask (3D bool)
    particle,                   # voxel particle mask (3D bool)
    voxel_size_m,               # float
    OUTPUT_DIR,                 # Path or str
    sample_name="sample",
    resolution_filter_nm=resolution_filter_nm,   # macropore threshold = equivalent diameter (nm)
    point_size=3,               # center marker size baseline
    export_html=True,           # save interactive HTML
    export_png=True,            # optional static export (needs kaleido)
    size_range_px=(2, 10),      # min/max dot size in pixels for Deq scaling
    size_scaling="sqrt",        # "linear" or "sqrt"
):
    OUTPUT_DIR = Path(OUTPUT_DIR)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    pores = np.asarray(pores, dtype=bool)
    particle = np.asarray(particle, dtype=bool)

    print("\nDetecting macropores (connected cavity clusters) from voxel mask...")

    # ---- Label connected pore clusters (3D) ----
    pore_labels, N_clusters = ndi.label(pores)
    print("Total pore clusters:", int(N_clusters))

    if N_clusters == 0:
        raise ValueError("No pore clusters found in 'pores' mask (pores.sum() == 0?).")

    # ---- Cluster sizes and equivalent diameters ----
    voxel_volume = float(voxel_size_m) ** 3

    cluster_sizes = ndi.sum(
        np.ones_like(pores, dtype=np.int8),
        labels=pore_labels,
        index=np.arange(1, N_clusters + 1)
    ).astype(float)

    cluster_volumes_m3 = cluster_sizes * voxel_volume

    # spherical equivalent diameter (descriptor)
    cluster_Deq_m = (6.0 * cluster_volumes_m3 / np.pi) ** (1.0 / 3.0)
    cluster_Deq_nm = cluster_Deq_m * 1e9

    # ---- Macropore threshold ----
    macro_ids = np.where(cluster_Deq_nm >= float(resolution_filter_nm))[0] + 1  # labels start at 1
    print(f"Macropores (Deq ≥ {float(resolution_filter_nm):.0f} nm):", int(macro_ids.size))

    if macro_ids.size == 0:
        raise ValueError(
            "No macropores detected with current threshold. "
            "Try lowering resolution_filter_nm (e.g., 200) or verify your pore mask."
        )

    # ---- Centroids (in voxel indices -> meters) ----
    centroids_vox = np.asarray(ndi.center_of_mass(pores, pore_labels, macro_ids), dtype=float)
    centroids_m = centroids_vox * float(voxel_size_m)

    # ---- Particle center & radius (r/R) ----
    center_particle_vox = np.asarray(ndi.center_of_mass(particle), dtype=float)
    center_particle_m = center_particle_vox * float(voxel_size_m)

    r_macro = np.linalg.norm(centroids_m - center_particle_m, axis=1)

    particle_coords_vox = np.argwhere(particle)
    if particle_coords_vox.size == 0:
        raise ValueError("Particle mask is empty (particle.sum() == 0?).")

    particle_coords_m = particle_coords_vox * float(voxel_size_m)
    R_particle_m = np.linalg.norm(particle_coords_m - center_particle_m, axis=1).max()
    R_particle_m = max(float(R_particle_m), 1e-12)

    r_over_R = r_macro / R_particle_m

# ============================================================
# Marker size varies with macropore Deq (nm)
# ============================================================
    macro_Deq_nm = cluster_Deq_nm[macro_ids - 1]  # one Deq per macropore in same order as centroids

    d = np.asarray(macro_Deq_nm, dtype=float)
    if not np.any(np.isfinite(d)):
        raise ValueError("macro_Deq_nm contains no finite values.")

    dmin = float(np.nanmin(d))
    dmax = float(np.nanmax(d))
    denom = (dmax - dmin) if (dmax - dmin) > 0 else 1.0
    d_norm = (d - dmin) / denom
    d_norm = np.clip(d_norm, 0.0, 1.0)

    if size_scaling.lower() == "sqrt":
        d_norm = np.sqrt(d_norm)
    elif size_scaling.lower() != "linear":
        raise ValueError("size_scaling must be 'linear' or 'sqrt'.")

    s_min, s_max = map(float, size_range_px)
    if s_min <= 0 or s_max <= 0 or s_max < s_min:
        raise ValueError("size_range_px must be (positive_min, positive_max) with max >= min.")

    marker_sizes = s_min + d_norm * (s_max - s_min)

# ============================================================
# Convert coordinates to micrometers for plotting
# ============================================================
    centroids_um = centroids_m * 1e6
    center_particle_um = center_particle_m * 1e6
    R_particle_um = R_particle_m * 1e6

# ============================================================
# LEFT: 3D interactive plot (Plotly)
# ============================================================
    out3d_html = OUTPUT_DIR / f"{sample_name}_3D_macropores_plotly.html"
    out3d_png = OUTPUT_DIR / f"{sample_name}_3D_macropores_plotly.png"

    x, y, z = centroids_um[:, 0], centroids_um[:, 1], centroids_um[:, 2]
    cx, cy, cz = center_particle_um

    # Wireframe sphere (envelope) as 3 great circles
    t = np.linspace(0, 2*np.pi, 361)
    circle_xy = np.c_[cx + R_particle_um*np.cos(t), cy + R_particle_um*np.sin(t), np.full_like(t, cz)]
    circle_xz = np.c_[cx + R_particle_um*np.cos(t), np.full_like(t, cy), cz + R_particle_um*np.sin(t)]
    circle_yz = np.c_[np.full_like(t, cx), cy + R_particle_um*np.cos(t), cz + R_particle_um*np.sin(t)]

    fig3d = go.Figure()

    fig3d.add_trace(
        go.Scatter3d(
            x=x, y=y, z=z,
            mode="markers",
            marker=dict(
                size=marker_sizes,
                color=r_over_R,
                colorscale="Magma",
                cmin=0, cmax=1,
                colorbar=dict(title=dict(text="r/R", side="right"), x=0.8, y=0.5, len=0.5),
                opacity=0.95,
            ),
            name="Macropore centroids",
            customdata=macro_Deq_nm,
            hovertemplate=(
                "x=%{x:.2f} µm<br>"
                "y=%{y:.2f} µm<br>"
                "z=%{z:.2f} µm<br>"
                "r/R=%{marker.color:.3f}<br>"
                "Deq=%{customdata:.1f} nm"
                "<extra></extra>"
            ),
        )
    )

    fig3d.add_trace(
        go.Scatter3d(
            x=[cx], y=[cy], z=[cz],
            mode="markers",
            marker=dict(size=max(point_size + 3, 6), symbol="circle", opacity=1.0),
            name="Particle center",
            hovertemplate="Particle center<extra></extra>",
        )
    )

    for circ, nm in [(circle_xy, "Envelope (xy)"), (circle_xz, "Envelope (xz)"), (circle_yz, "Envelope (yz)")]:
        fig3d.add_trace(
            go.Scatter3d(
                x=circ[:, 0], y=circ[:, 1], z=circ[:, 2],
                mode="lines",
                line=dict(width=2),
                opacity=0.35,
                name=nm,
                hoverinfo="skip",
            )
        )

    fig3d.update_layout(
        title=f"{sample_name} — Macropores (Deq ≥ {float(resolution_filter_nm):.0f} nm)",
        scene=dict(
            xaxis=dict(title="x (µm)"),
            yaxis=dict(title="y (µm)"),
            zaxis=dict(title="z (µm)"),
            aspectmode="data",
            camera=dict(eye=dict(x=1.25, y=1.25, z=1.25))
        ),
        legend=dict(itemsizing="constant"),
        margin=dict(l=0, r=0, t=50, b=0),
    )

    if export_html:
        fig3d.write_html(str(out3d_html), include_plotlyjs="cdn")
    if export_png:
        fig3d.write_image(str(out3d_png), scale=2, engine="kaleido")
        print("PNG written to:", out3d_png.resolve())

# ==============================================================
# RIGHT: 2D distribution of radial positions (r/R) -> matplotlib
# ==============================================================
    out_dist = OUTPUT_DIR / f"{sample_name}_macropore_rOverR_distribution.png"
    plt.figure(figsize=(6, 4))
    plt.hist(r_over_R, bins=30, density=True, edgecolor="black", linewidth=0.6)
    plt.xlabel("Normalized radial position, r/R")
    plt.ylabel("Probability density")
    plt.title("Macropore centroid distance distribution")
    plt.xlim(0, 1)
    plt.tight_layout()
    plt.savefig(out_dist, dpi=300, bbox_inches="tight")
    plt.close()

    print("\nSaved:")
    if export_html:
        print(" - 3D macropores (interactive):", out3d_html.resolve())
    if export_png:
        print(" - 3D macropores (static PNG):", out3d_png.resolve())
    print(" - r/R distribution:", out_dist.resolve())

    return {
        "macro_ids": macro_ids,
        "centroids_m": centroids_m,
        "centroids_um": centroids_um,
        "r_over_R": r_over_R,
        "cluster_Deq_nm": cluster_Deq_nm,
        "macro_Deq_nm": macro_Deq_nm,
        "marker_sizes": marker_sizes,
        "R_particle_m": R_particle_m,
        "R_particle_um": R_particle_um,
        "center_particle_m": center_particle_m,
        "center_particle_um": center_particle_um,
        "out3d_html": out3d_html if export_html else None,
        "out3d_png": out3d_png if export_png else None,
        "out_dist": out_dist,
        "fig3d": fig3d,
    }


results = make_main_panel_macropores_plotly(
    pores=pores,
    particle=particle,
    voxel_size_m=voxel_size_m,
    OUTPUT_DIR=OUTPUT_DIR,
    sample_name="Cu_cop_spent",
    resolution_filter_nm=resolution_filter_nm,
    point_size=2,
    size_range_px=(2, 12),
    size_scaling="sqrt",
    export_html=True,
    export_png=True,
)


#### Step 9 - Pore centroids to surface distance: approach (1)

In [ ]:
# In this case we approximate the particle to a sphere with certain radius and use this radius as R.
# r(i) is the euclidean distance from the centroid of each pore (i) to the center of the spherical approximation of our particle.
# r(i)/R is the relative distance we are plotting as distribution.

In [ ]:
def plot_distance_panel_plotly_static(
    centroids_um,
    r_over_R,
    marker_sizes,
    particle_radius_um=None,          # optional: if you want an absolute-radius “shell”
    sample_name="sample",
    bins=64,
    camera_eye=dict(x=1.5, y=1.5, z=0.8),
    show_particle_shell=True,         # draw transparent particle boundary
    shell_opacity=0.08,               # boundary transparency
    shell_resolution=10,              # mesh resolution (higher = smoother, heavier)
    shell_color="olive",              # neutral boundary color
    shell_center_um=None,             # (3,) center. If None, infer from centroids
    equal_axes=True,                  # enforce equal x/y/z ranges (nice for publication)
):
    centroids_um = np.asarray(centroids_um, dtype=float)
    r_over_R = np.asarray(r_over_R, dtype=float)

    if centroids_um.ndim != 2 or centroids_um.shape[1] != 3:
        raise ValueError("centroids_um must be an array of shape (N, 3).")
    if r_over_R.shape[0] != centroids_um.shape[0]:
        raise ValueError("r_over_R must have same length as centroids_um.")
    if np.asarray(marker_sizes).shape[0] != centroids_um.shape[0]:
        raise ValueError("marker_sizes must have same length as centroids_um.")

    x, y, z = centroids_um[:, 0], centroids_um[:, 1], centroids_um[:, 2]

    # ------------------------------------------------------------
    # Infer particle center if not provided (robust: use median)
    # ------------------------------------------------------------
    if shell_center_um is None:
        cx, cy, cz = np.median(x), np.median(y), np.median(z)
    else:
        shell_center_um = np.asarray(shell_center_um, dtype=float)
        if shell_center_um.shape != (3,):
            raise ValueError("shell_center_um must be a length-3 iterable (cx, cy, cz).")
        cx, cy, cz = shell_center_um

    # ------------------------------------------------------------
    # Infer radius if not provided:
    # - if particle_radius_um is None, use max distance of points from center
    #   (works as a visual boundary; not a true particle radius unless center is correct)
    # ------------------------------------------------------------
    if particle_radius_um is None:
        rr = np.sqrt((x - cx) ** 2 + (y - cy) ** 2 + (z - cz) ** 2)
        R_um = float(np.nanmax(rr)) if rr.size else 1.0
    else:
        R_um = float(particle_radius_um)

    # ------------------------------------------------------------
    # Build subplot canvas
    # ------------------------------------------------------------
    fig = make_subplots(
        rows=1,
        cols=2,
        specs=[[{"type": "scene"}, {"type": "xy"}]],
        column_widths=[0.60, 0.40],
        horizontal_spacing=0.04,
    )

    # ============================================================
    # LEFT → Particle boundary shell 
    # ============================================================
    if show_particle_shell and np.isfinite(R_um) and R_um > 0:
        # Parametric sphere mesh (u,v)
        u = np.linspace(0, 2 * np.pi, shell_resolution)
        v = np.linspace(0, np.pi, shell_resolution)
        uu, vv = np.meshgrid(u, v)

        xs = cx + R_um * np.cos(uu) * np.sin(vv)
        ys = cy + R_um * np.sin(uu) * np.sin(vv)
        zs = cz + R_um * np.cos(vv)

        fig.add_trace(
            go.Surface(
                x=xs,
                y=ys,
                z=zs,
                showscale=False,
                opacity=shell_opacity,
                colorscale=[[0, shell_color], [1, shell_color]],
                hoverinfo="skip",
            ),
            row=1,
            col=1,
        )

    # ============================================================
    # LEFT → 3D centroids
    # ============================================================
    fig.add_trace(
        go.Scatter3d(
            x=x, y=y, z=z,
            mode="markers",
            marker=dict(
                size=marker_sizes,
                color=r_over_R,
                colorscale="magma",
                cmin=0, cmax=1,
                opacity=1,
                colorbar=dict(
                    title=dict(text="r/R", font=dict(family="Arial", size=18)),
                    len=0.75,
                    thickness=18,
                    x=0.40,
                    y=0.55,
                    tickfont=dict(family="Arial", size=16),
                ),
            ),
            showlegend=False,
        ),
        row=1,
        col=1,
    )

    # ============================================================
    # RIGHT → Histogram
    # ============================================================
    fig.add_trace(
        go.Histogram(
            x=r_over_R,
            nbinsx=bins,
            histnorm="probability density",
            marker=dict(color="#E67E22", line=dict(width=0)),
            opacity=0.95,
            showlegend=False,
        ),
        row=1,
        col=2,
    )

    # ============================================================
    # Nice 3D axis ranges 
    # ============================================================
    scene_axes = dict(
        xaxis=dict(title="x (µm)", showgrid=True, nticks=3),
        yaxis=dict(title="y (µm)", showgrid=True, nticks=5),
        zaxis=dict(title="z (µm)", showgrid=True, nticks=5),
    )

    if equal_axes:
        # Make x/y/z ranges equal around the chosen center, based on radius R_um
        pad = 0.05 * R_um
        scene_axes["xaxis"]["range"] = [cx - R_um - pad, cx + R_um + pad]
        scene_axes["yaxis"]["range"] = [cy - R_um - pad, cy + R_um + pad]
        scene_axes["zaxis"]["range"] = [cz - R_um - pad, cz + R_um + pad]

    # ============================================================
    # Layout tuned for publication
    # ============================================================
    fig.update_layout(
        width=1000,
        height=480,
        font=dict(family="Arial", size=14, color="black"),
        plot_bgcolor="white",
        margin=dict(l=10, r=10, t=60, b=10),

        scene=dict(
            **scene_axes,
            aspectmode="data",
            camera=dict(eye=camera_eye),
            domain=dict(x=[0.0, 0.45], y=[0.15, 0.99]),
        ),

        xaxis=dict(domain=[0.58, 0.98]),
        yaxis=dict(domain=[0.3, 0.87]),
    )

    # ============================================================
    # Histogram axes styling (matplotlib-like)
    # ============================================================
    fig.update_xaxes(
        title=dict(text="Normalized radial position (r/R)", font=dict(family="Arial", size=18)),
        range=[0, 1],
        showgrid=False,
        showline=True,
        linecolor="black",
        linewidth=1,
        ticks="outside",
        ticklen=6,
        mirror=True,
        row=1,
        col=2,
        tickfont=dict(family="Arial", size=16),
    )

    fig.update_yaxes(
        title=dict(text="Probability density", font=dict(family="Arial", size=18)),
        showgrid=False,
        showline=True,
        linecolor="black",
        linewidth=1,
        ticks="outside",
        ticklen=6,
        mirror=True,
        row=1,
        col=2,
        tickfont=dict(family="Arial", size=16),
    )

    return fig


# ---------------------------
# Usage
# ---------------------------
fig = plot_distance_panel_plotly_static(
    centroids_um=results["centroids_um"],
    r_over_R=results["r_over_R"],
    marker_sizes=results["marker_sizes"],
    # particle_radius_um=... ,   # optionally set your known particle radius
    show_particle_shell=True,
    shell_opacity=0.08,
    shell_resolution=45,
    equal_axes=True,
)

fig.show()

fig.write_image(
    OUTPUT_DIR / f"{sample_name}_DISTANCE_PANEL.png",
    scale=2,
    engine="kaleido",
)

#### Step 10 - Pore centroids to surface distances: approach (2)

In [ ]:
## In this block of code another approach is used: 

#    For each macropore cluster:
#      1) compute centroid (center of mass) in voxel coords
#      2) compute distance-to-real-surface at that centroid using distance_transform_edt(particle)
#      3) plot histogram of distances (µm)

#    Returns:
#      d_surf_um: (N_macro,) distances from pore centroids to particle surface [µm]
#      centroids_vox: (N_macro,3) centroid coordinates [voxels]

In [ ]:
def centroid_distance_to_surface_distribution(
    pores,                      # 3D bool pore mask
    particle,                   # 3D bool particle mask (True = inside particle)
    voxel_size_m,               # float
    OUTPUT_DIR,                 # Path or str
    sample_name="sample",
    resolution_filter_nm=200,   # macropore threshold (nm) based on Deq
    bins=40,
    export_png=True,
):
    """
    For each macropore cluster:
      1) compute centroid (center of mass) in voxel coords
      2) compute distance-to-real-surface at that centroid using distance_transform_edt(particle)
      3) plot histogram of distances (µm)

    Returns:
      d_surf_um: (N_macro,) distances from pore centroids to particle surface [µm]
      centroids_vox: (N_macro,3) centroid coordinates [voxels]
      macro_ids: (N_macro,) pore label IDs
    """
    OUTPUT_DIR = Path(OUTPUT_DIR)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    pores = np.asarray(pores, dtype=bool)
    particle = np.asarray(particle, dtype=bool)

    if pores.shape != particle.shape:
        raise ValueError("pores and particle must have the same shape.")
    if particle.sum() == 0:
        raise ValueError("Particle mask is empty (particle.sum() == 0?).")

    # ---- Label connected pore clusters ----
    pore_labels, N = ndi.label(pores)
    if N == 0:
        raise ValueError("No pore clusters found in pores mask.")

    # ---- Equivalent diameter to filter macropores ----
    voxel_volume = float(voxel_size_m) ** 3
    cluster_sizes = ndi.sum(
        np.ones_like(pores, dtype=np.int8),
        labels=pore_labels,
        index=np.arange(1, N + 1),
    ).astype(float)
    cluster_vol_m3 = cluster_sizes * voxel_volume
    cluster_Deq_m = (6.0 * cluster_vol_m3 / np.pi) ** (1.0 / 3.0)
    cluster_Deq_nm = cluster_Deq_m * 1e9

    macro_ids = np.where(cluster_Deq_nm >= float(resolution_filter_nm))[0] + 1
    if macro_ids.size == 0:
        raise ValueError("No macropores detected with current threshold.")

    # ---- Centroids (float voxel coordinates) ----
    centroids_vox = np.asarray(ndi.center_of_mass(pores, pore_labels, macro_ids), dtype=float)

    # ---- Distance-to-surface map (REAL particle boundary) ----
    # dt(voxel) is distance (in voxels) to nearest outside-of-particle voxel (boundary)
    dt_vox = ndi.distance_transform_edt(particle)
    dt_um = dt_vox * float(voxel_size_m) * 1e6

    # ---- Sample dt at centroid locations ----
    d_surf_um = ndi.map_coordinates(
        dt_um,
        [centroids_vox[:, 0], centroids_vox[:, 1], centroids_vox[:, 2]],
        order=1,
        mode="nearest",
    ).astype(float)

    d_surf_um = d_surf_um[np.isfinite(d_surf_um)]

    # ---- Plot histogram ----
    out_png = OUTPUT_DIR / f"{sample_name}_macropore_centroid_distance_to_surface.png"
    plt.figure(figsize=(6, 4))
    plt.hist(d_surf_um, bins=bins, density=True, edgecolor="black", linewidth=0.6)
    plt.xlabel("Centroid distance to particle surface (µm)")
    plt.ylabel("Probability density")
    plt.title(f"{sample_name} — Macropore centroid distance to real surface")
    plt.tight_layout()

    if export_png:
        plt.savefig(out_png, dpi=300, bbox_inches="tight")
        print("Saved:", out_png.resolve())
    plt.close()

    print("N macropores:", d_surf_um.size)
    print("Centroid distance-to-surface (µm): median =", float(np.median(d_surf_um)),
          " mean =", float(np.mean(d_surf_um)),
          " min =", float(np.min(d_surf_um)),
          " max =", float(np.max(d_surf_um)))

    return d_surf_um, centroids_vox, macro_ids


# ---------------------------
# Usage
# ---------------------------
d_surf_um, centroids_vox, macro_ids = centroid_distance_to_surface_distribution(
    pores=pores,
    particle=particle,
    voxel_size_m=voxel_size_m,
    OUTPUT_DIR=OUTPUT_DIR,
    sample_name="Cu_cop_spent",
    resolution_filter_nm=resolution_filter_nm,
    bins=40,
    export_png=True,
)

#### Step 11 - Pore centroids to surface distances: plots for approach (2)

In [ ]:
# Calculates the distance from each pore centroid and the minimal distance from the surface of the particle.
# The given distance is distributed in a histogram and colors for distribution

In [ ]:
# -----------------------------------------
# Convert centroids to micrometers
# -----------------------------------------
centroids_um = centroids_vox * voxel_size_m * 1e6

x = centroids_um[:,0]
y = centroids_um[:,1]
z = centroids_um[:,2]

color_values = d_surf_um   # centroid distance to real particle surface


# -----------------------------------------
# Create panel (3D + histogram)
# -----------------------------------------
fig = make_subplots(
    rows=1,
    cols=2,
    specs=[[{"type": "scene"}, {"type": "xy"}]],
    column_widths=[0.60,0.40],
    horizontal_spacing=0.04
)


# -----------------------------------------
# 3D pore centroid map
# -----------------------------------------
fig.add_trace(
    go.Scatter3d(
        x=x,
        y=y,
        z=z,
        mode="markers",
        marker=dict(
            size=results["marker_sizes"],
            color=color_values,
            colorscale="magma",
            opacity=1,
            colorbar=dict(
                title=dict(text="D (µm)", font=dict(family="Arial", size=18)),
                len=0.75,
                thickness=18,
                x=0.40,
                y=0.55,
                tickfont=dict(family="Arial", size=16)
            )
        ),
        showlegend=False
    ),
    row=1,
    col=1
)


# -----------------------------------------
# Histogram
# -----------------------------------------
fig.add_trace(
    go.Histogram(
        x=color_values,
        nbinsx=60,
        histnorm="probability density",
        marker=dict(color="#E67E22", line=dict(width=0)),
        opacity=0.95,
        showlegend=False
    ),
    row=1,
    col=2
)


# -----------------------------------------
# Layout styling (MATCHES YOUR OTHER PLOT)
# -----------------------------------------
fig.update_layout(
    width=1000,
    height=480,
    font=dict(family="Arial", size=14, color="black"),
    plot_bgcolor="white",
    margin=dict(l=10,r=10,t=60,b=10),

    scene=dict(
        xaxis=dict(title="x (µm)", showgrid=True, nticks=3),
        yaxis=dict(title="y (µm)", showgrid=True, nticks=5),
        zaxis=dict(title="z (µm)", showgrid=True, nticks=5),
        aspectmode="data",
        camera=dict(eye=dict(x=1.5,y=1.5,z=1.5)),
        domain=dict(x=[0.0,0.45],y=[0.15,0.99])
    ),

    xaxis=dict(domain=[0.58,0.98]),
    yaxis=dict(domain=[0.3,0.87])
)


# -----------------------------------------
# Histogram axis styling (MATCHES OTHER PLOT)
# -----------------------------------------
fig.update_xaxes(
    title=dict(text="Pore centroid distance to particle surface (µm)", font=dict(family="Arial", size=18)),
    showgrid=False,
    showline=True,
    linecolor="black",
    linewidth=1,
    ticks="outside",
    ticklen=6,
    mirror=True,
    row=1,
    col=2,
    tickfont=dict(family="Arial", size=16)
)

fig.update_yaxes(
    title=dict(text="Probability density", font=dict(family="Arial", size=18)),
    showgrid=False,
    showline=True,
    linecolor="black",
    linewidth=1,
    ticks="outside",
    ticklen=6,
    mirror=True,
    row=1,
    col=2,
    tickfont=dict(family="Arial", size=16)
)


# -----------------------------------------
# Show
# -----------------------------------------
fig.show()


# -----------------------------------------
# Save
# -----------------------------------------
fig.write_image(
    OUTPUT_DIR / f"{sample_name}_DISTANCE_TO_SURFACE_PANEL.png",
    scale=2,
    engine="kaleido"
)




### Step 12 - Pore centroids to surface distance: normalized approach (2) 

In [ ]:
# The normalization is done by choosing the pore that is deeper inside the particle. 
# All other pore distances are divided by the maximum. 
# 1-d (pore centroid i distance)/Dmax = Proximity (a.u.)

In [ ]:
# =========================================================
# OPTION A (standalone plot block)
# =========================================================

# --- Convert centroids to µm ---
centroids_um = np.asarray(centroids_vox, dtype=float) * float(voxel_size_m) * 1e6
x, y, z = centroids_um[:, 0], centroids_um[:, 1], centroids_um[:, 2]

# --- Option A normalization (relative proximity to surface) ---
# NOTE: Ideally Dmax comes from the particle distance-transform max.
# If plotting separately and only have d_surf_um, this is a practical proxy.
Dmax_um = float(np.nanmax(d_surf_um))
Dmax_um = max(Dmax_um, 1e-12)

surface_proximity = 1.0 - (np.asarray(d_surf_um, dtype=float) / Dmax_um)
surface_proximity = np.clip(surface_proximity, 0.0, 1.0)

# --- For "equal_axes" framing (same as your function) ---
cx, cy, cz = np.median(x), np.median(y), np.median(z)
rr = np.sqrt((x - cx) ** 2 + (y - cy) ** 2 + (z - cz) ** 2)
R_um = float(np.nanmax(rr)) if rr.size else 1.0
pad = 0.05 * R_um

# --- Build subplot canvas (same geometry) ---
fig = make_subplots(
    rows=1,
    cols=2,
    specs=[[{"type": "scene"}, {"type": "xy"}]],
    column_widths=[0.60, 0.40],
    horizontal_spacing=0.04,
)

# ============================================================
# LEFT → 3D centroids colored by Option A surface proximity
# ============================================================
fig.add_trace(
    go.Scatter3d(
        x=x, y=y, z=z,
        mode="markers",
        marker=dict(
            size=results["marker_sizes"],
            color=surface_proximity,
            colorscale="magma",
            cmin=0, cmax=1,
            opacity=1,
            colorbar=dict(
                title=dict(text="Proximity (a.u.)", font=dict(family="Arial", size=20)),
                len=0.75,
                thickness=20,
                x=0.40,
                y=0.55,
                tickfont=dict(family="Arial", size=18),
            ),
        ),
        showlegend=False,
    ),
    row=1,
    col=1,
)

# ============================================================
# RIGHT → Histogram
# ============================================================
fig.add_trace(
    go.Histogram(
        x=surface_proximity,
        nbinsx=64,
        histnorm="probability density",
        marker=dict(color="#E67E22", line=dict(width=0)),
        opacity=0.95,
        showlegend=False
    ),
    row=1,
    col=2,
)

# ============================================================
# Layout tuned for publication 
# ============================================================
fig.update_layout(
    width=1000,
    height=480,
    font=dict(family="Arial", size=16, color="black"),
    plot_bgcolor="white",
    margin=dict(l=10, r=10, t=60, b=10),

    scene=dict(
        xaxis=dict(title="x (µm)", showgrid=True, nticks=3, range=[cx - R_um - pad, cx + R_um + pad]),
        yaxis=dict(title="y (µm)", showgrid=True, nticks=5, range=[cy - R_um - pad, cy + R_um + pad]),
        zaxis=dict(title="z (µm)", showgrid=True, nticks=5, range=[cz - R_um - pad, cz + R_um + pad]),
        aspectmode="data",
        camera=dict(eye=dict(x=1.5, y=1.5, z=1.5)),
        domain=dict(x=[0.0, 0.45], y=[0.15, 0.99]),
    ),

    xaxis=dict(domain=[0.58, 0.98]),
    yaxis=dict(domain=[0.3, 0.87]),
)

# ============================================================
# Histogram axes styling 
# ============================================================
fig.update_xaxes(
    title=dict(text="Proximity (0 = deepest, 1 = surface)", font=dict(family="Arial", size=20)),
    range=[0, 1],
    showgrid=False,
    showline=True,
    linecolor="black",
    linewidth=1,
    ticks="outside",
    ticklen=6,
    mirror=True,
    row=1,
    col=2,
    tickfont=dict(family="Arial", size=20),
)

fig.update_yaxes(
    title=dict(text="Probability density", font=dict(family="Arial", size=22)),
    showgrid=False,
    showline=True,
    linecolor="black",
    linewidth=1,
    ticks="outside",
    ticklen=6,
    mirror=True,
    row=1,
    col=2,
    tickfont=dict(family="Arial", size=20),
)

# --- Show & save ---
fig.show()

fig.write_image(
    OUTPUT_DIR / f"{sample_name}_SURFACE_PROXIMITY_PANEL.png",
    scale=5,
    engine="kaleido",
)